# Sesión 22 — Cierre: Presentaciones de Proyecto y Retrospectiva del Curso
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo VI · Integración, Ética y Cierre**

## Estructura de la sesión (2.5 horas)

| Segmento | Duración | Contenido |
|---|---|---|
| Presentaciones de proyecto | 90 min | 5–6 min por equipo + 3 min de preguntas |
| Taller de crítica entre pares | 20 min | Retroalimentación estructurada sobre dos proyectos asignados |
| Discusión de problemas abiertos | 20 min | Qué permanece sin resolver en ML biomédico |
| Retrospectiva del curso | 20 min | Qué funcionó, qué llevarse |

Este cuaderno cumple tres propósitos:
1. **Andamiaje de evaluación de proyecto** — una plantilla para crítica entre pares y autoevaluación
2. **Lista de verificación del pipeline del proyecto final** — verificar que todos los componentes requeridos estén presentes
3. **Galería de problemas abiertos** — direcciones de investigación curadas para que los estudiantes continúen explorando

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(0)
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})
print('Cuaderno de Sesión 22 — Cierre cargado.')

## Parte 1 — Pipeline del proyecto final: componentes requeridos

Un proyecto final completo debe demostrar el pipeline completo de ML. Usa esta lista
de verificación para autoevaluación y revisión por pares.

In [ ]:
# ── Rúbrica de autoevaluación del proyecto final ──────────────────────────────
# Completa con las puntuaciones reales de tu proyecto (escala 0–4) para cada criterio

secciones_rubrica = {
    'Planteamiento del problema\ny datos (20 pts)': {
        'Pregunta clínica claramente planteada':         4,
        'Dataset descrito (tamaño, fuente, calidad)':    4,
        'Balance de clases y datos faltantes abordados': 3,
        'Consideraciones éticas/privacidad discutidas':  3,
        'Métrica de evaluación justificada clínicamente': 3,
    },
    'Preprocesamiento y\ncaracterísticas (20 pts)': {
        'Normalización apropiada aplicada':              4,
        'Extracción de características o e2e justificada': 3,
        'Prevención de fuga de datos documentada':       4,
        'Aumentación o manejo de desbalance':            2,
        'Pipeline de preprocesamiento reproducible':     3,
    },
    'Modelado (25 pts)': {
        'Al menos 3 modelos comparados':                 4,
        'Estrategia de CV apropiada (LOSO si aplica)':   4,
        'Búsqueda de hiperparámetros documentada':       3,
        'Mejor modelo seleccionado con justificación':   4,
        'Curvas de entrenamiento mostradas e interpretadas': 3,
        'Incertidumbre/confianza reportada':             2,
    },
    'Evaluación e\ninterpretación (20 pts)': {
        'AUROC con IC 95% reportado':                    4,
        'Matriz de confusión y métricas por clase':      4,
        'Calibración evaluada':                          3,
        'Explicabilidad (Grad-CAM, SHAP, etc.)':         3,
        'Análisis de subgrupos / verificación de equidad': 2,
    },
    'Presentación y\nreproducibilidad (15 pts)': {
        'Código limpio y bien comentado':                3,
        'README con instrucciones de configuración':     3,
        'Tablas/figuras de resultados listas para publicación': 3,
        'Limitaciones claramente declaradas':            3,
        'Sugerencias para trabajo futuro':                3,
    },
}

# Calcular puntuaciones
puntajes_seccion, maximos_seccion = {}, {}
for seccion, criterios in secciones_rubrica.items():
    score  = sum(criterios.values())
    max_sc = len(criterios) * 4
    puntajes_seccion[seccion] = score
    maximos_seccion[seccion]  = max_sc

total = sum(puntajes_seccion.values())
max_t = sum(maximos_seccion.values())

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico de barras estilo radar
secciones = list(puntajes_seccion.keys())
pcts      = [100*puntajes_seccion[s]/maximos_seccion[s] for s in secciones]
colores_r = ['steelblue','darkorange','seagreen','tomato','mediumpurple']

axes[0].barh(secciones, pcts, color=colores_r, alpha=0.8, edgecolor='white')
axes[0].axvline(75, color='gray', ls='--', lw=1.5, label='Objetivo 75%')
for i, (pct, sec) in enumerate(zip(pcts, secciones)):
    pts = puntajes_seccion[sec]
    mx  = maximos_seccion[sec]
    axes[0].text(pct + 1, i, f'{pts}/{mx} pts', va='center', fontsize=9)
axes[0].set(xlim=(0, 115), xlabel='Puntuación (%)', title='Puntuaciones de la rúbrica del proyecto final')
axes[0].legend(fontsize=8)

# Pastel general
grade_pct = 100 * total / max_t
axes[1].pie(
    [total, max_t - total],
    labels=[f'Obtenido\n{total}/{max_t}', f'Restante\n{max_t-total}/{max_t}'],
    colors=['steelblue', '#e8e8e8'],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title(f'Puntuación general: {grade_pct:.0f}%\n'
                   f'Banda de calificación: {"A" if grade_pct>=90 else "B" if grade_pct>=80 else "C" if grade_pct>=70 else "D"}')

plt.suptitle('Autoevaluación del proyecto final (modifica las puntuaciones según tu proyecto)', y=1.01)
plt.tight_layout()
plt.show()

## Parte 2 — Plantilla estructurada de crítica entre pares

Para cada proyecto asignado, completa lo siguiente. El objetivo es la especificidad
constructiva — no "los resultados fueron buenos" sino "el AUROC de 0.89 es competitivo
con el benchmark de la literatura de 0.87 de Smith et al. 2021, aunque el intervalo de
confianza es amplio debido al conjunto de test pequeño."

In [ ]:
# Plantilla de crítica entre pares — completar durante las presentaciones
plantilla_critica = """
==================================================
CRÍTICA ENTRE PARES — Proyecto: [INSERTAR TÍTULO DEL PROYECTO]
Revisor: [TU NOMBRE]   Fecha: [FECHA]
==================================================

1. ASPECTO MÁS SÓLIDO
   ¿Qué hizo este proyecto particularmente bien? Sé específico.
   > [ej., "La validación cruzada LOSO se implementó correctamente, evitando
     fuga de información a nivel de sujeto que habría inflado el AUROC en
     ~0.12 en este dataset."]

2. UNA PREOCUPACIÓN METODOLÓGICA
   Identifica una decisión que podría afectar la validez de los resultados.
   > [ej., "La selección de características se aplicó antes del ciclo de CV
     (líneas 45–52), lo cual constituye fuga de datos. Impacto esperado:
     AUROC sobreestimado en ~0.05–0.10."]

3. COMPONENTE FALTANTE
   ¿Qué análisis o verificación importante no se incluyó?
   > [ej., "No se evaluó la calibración. Dado el desbalance de clases (8%
     positivos), un modelo puede parecer bien calibrado por AUROC mientras
     está mal calibrado en la región de baja probabilidad, la más relevante
     clínicamente."]

4. UNA SUGERENCIA CONCRETA
   Si tuvieras una semana más, ¿qué priorizarías?
   > [ej., "Añadir análisis SHAP específicamente para las 5 características
     principales. La importancia de características global actual no revela
     efectos de interacción ni la dirección de influencia para predicciones
     individuales."]

5. TRASLADABILIDAD CLÍNICA
   ¿Sería desplegable este sistema en un entorno clínico real? ¿Qué falta?
   > [ej., "Falta validación externa en una cohorte de otro hospital. Los
     conjuntos de entrenamiento y test son de la misma institución, por lo
     que no puede asumirse generalización."]

IMPRESIÓN GENERAL (1–5):
   Rigor metodológico:        [  ]
   Relevancia clínica:        [  ]
   Claridad de presentación:  [  ]
   Reproducibilidad del código: [  ]
==================================================
"""
print(plantilla_critica)

## Parte 3 — Problemas abiertos en ML biomédico

Una lista curada de direcciones de investigación que el campo aún no ha resuelto —
cualquiera de las cuales podría convertirse en una tesis doctoral o un artículo.

In [ ]:
problemas_abiertos = [
    {
        'area': 'Generalización',
        'problema': 'Generalización entre sitios sin compartir datos',
        'por_que_dificil': 'Ruido específico del sitio, diferencias de escáner, '
                           'deriva demográfica, variación de protocolo — todo confundido junto',
        'direcciones_prometedoras': 'Aprendizaje federado + adaptación de dominio; '
                                    'aprendizaje de representación causal; adaptación en tiempo de prueba',
        'articulo_clave': 'Guan et al. (2021) Domain adaptation for medical imaging',
    },
    {
        'area': 'Escasez de etiquetas',
        'problema': 'Aprender de un puñado de ejemplos biomédicos etiquetados',
        'por_que_dificil': 'El meta-aprendizaje few-shot estándar asume similitud de tareas; '
                           'las tareas biomédicas son altamente heterogéneas',
        'direcciones_prometedoras': 'Ajuste fino de modelos fundacionales; '
                                    'SSL contrastivo + sondeo lineal; aprendizaje activo',
        'articulo_clave': 'Chen et al. (2022) Benchmarking few-shot learning in EHR',
    },
    {
        'area': 'Dinámica temporal',
        'problema': 'Modelar trayectorias de pacientes a largo plazo (años, no minutos)',
        'por_que_dificil': 'Muestreo irregular, visitas faltantes, riesgos competitivos, '
                           'intervenciones confunden las observaciones',
        'direcciones_prometedoras': 'ODEs neuronales; procesos de punto temporal; '
                                    'transformers de tiempo continuo',
        'articulo_clave': 'Rubanova et al. (2019) Latent ODEs for irregularly sampled time series',
    },
    {
        'area': 'Causalidad',
        'problema': 'Predecir el efecto de intervenciones (no solo asociaciones)',
        'por_que_dificil': 'Los datos observacionales tienen confusión; los datos de ECA son '
                           'pequeños y costosos',
        'direcciones_prometedoras': 'Inferencia causal + ML; predicción contrafactual; '
                                    'métodos de variable instrumental',
        'articulo_clave': 'Prosperi et al. (2020) Causal inference and counterfactual prediction in ML',
    },
    {
        'area': 'Fusión multimodal',
        'problema': 'Combinar EEG, fMRI, notas clínicas, genómica de forma coherente',
        'por_que_dificil': 'Distintas modalidades tienen distinto ruido, resolución, '
                           'patrones de datos faltantes; no hay consenso sobre arquitectura de fusión',
        'direcciones_prometedoras': 'Atención cruzada entre modalidades; tokens agnósticos '
                                    'de modalidad; fusión tardía con incertidumbre',
        'articulo_clave': 'Zhang et al. (2022) Multi-modal learning for clinical prediction',
    },
    {
        'area': 'Incertidumbre',
        'problema': 'Saber cuándo el modelo no sabe',
        'por_que_dificil': 'Calibración, incertidumbre epistémica vs aleatoria, '
                           'detección fuera de distribución requieren herramientas distintas',
        'direcciones_prometedoras': 'Ensambles profundos; predicción conforme; '
                                    'redes neuronales bayesianas',
        'articulo_clave': 'Kompa et al. (2021) Second opinion needed: communicating uncertainty in ML',
    },
    {
        'area': 'Aprendizaje continuo',
        'problema': 'Actualizar modelos conforme llegan nuevos datos sin olvidar',
        'por_que_dificil': 'Olvido catastrófico; implicaciones regulatorias de actualizaciones '
                           'de modelo; fisiología no estacionaria',
        'direcciones_prometedoras': 'Consolidación elástica de pesos; '
                                    'búferes de repetición; control de cambio predeterminado de la FDA',
        'articulo_clave': 'Bosnjak et al. (2021) Continual learning in medicine',
    },
]

colores_area = {
    'Generalización': 'steelblue', 'Escasez de etiquetas': 'darkorange',
    'Dinámica temporal': 'seagreen', 'Causalidad': 'tomato',
    'Fusión multimodal': 'mediumpurple', 'Incertidumbre': 'saddlebrown',
    'Aprendizaje continuo': 'teal',
}

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 10); ax.set_ylim(0, len(problemas_abiertos)+0.5)
ax.axis('off')

for i, prob in enumerate(problemas_abiertos):
    y = len(problemas_abiertos) - i - 0.5
    color = colores_area.get(prob['area'], 'gray')
    ax.add_patch(mpatches.FancyBboxPatch((0, y-0.35), 1.7, 0.7,
                                          boxstyle='round,pad=0.05',
                                          facecolor=color, alpha=0.8))
    ax.text(0.85, y, prob['area'], ha='center', va='center',
             fontsize=7, color='white', fontweight='bold')
    ax.text(1.85, y+0.18, prob['problema'], fontsize=8.5, va='center', fontweight='bold')
    dirs = prob['direcciones_prometedoras']
    texto_dir = f"→ {dirs[:80]}..." if len(dirs) > 80 else f"→ {dirs}"
    ax.text(1.85, y-0.15, texto_dir, fontsize=7.5, va='center', color='dimgray', style='italic')

ax.set_title('Problemas abiertos en ML biomédico — direcciones de investigación para la próxima década',
              fontsize=12, pad=10)
plt.tight_layout()
plt.show()

## Parte 4 — Retrospectiva del curso: el arco completo

In [ ]:
# Mapa del curso: las 22 sesiones ubicadas en dos ejes
# x: nivel de abstracción (estadístico → profundo)
# y: alcance de modalidad de datos (tabular → señal cruda → generativo)

mapa_sesiones = [
    # (sesión, x, y, módulo, etiqueta_corta)
    (1,  1.0, 3.0, 'I',   'S01 Probabilidad'),
    (2,  1.2, 3.2, 'I',   'S02 Bayesiana'),
    (3,  1.4, 2.8, 'I',   'S03 AUROC'),
    (4,  1.5, 2.6, 'I',   'S04 Validación'),
    (5,  1.8, 4.0, 'I',   'S05 PCA/ICA'),
    (6,  2.2, 2.5, 'II',  'S06 Logística'),
    (7,  2.4, 3.5, 'II',  'S07 SVM'),
    (8,  2.6, 2.2, 'II',  'S08 Ensambles'),
    (9,  2.8, 2.8, 'II',  'S09 Explicabilidad'),
    (10, 3.2, 4.2, 'III', 'S10 GMM/EM'),
    (11, 3.4, 4.5, 'III', 'S11 HMM'),
    (12, 4.2, 3.0, 'IV',  'S12 MLP/Backprop'),
    (13, 5.0, 5.0, 'IV',  'S13 CNN'),
    (14, 5.2, 4.8, 'IV',  'S14 RNN/LSTM'),
    (15, 4.8, 3.5, 'IV',  'S15 Regularización'),
    (16, 6.0, 4.5, 'V',   'S16 Transformers'),
    (17, 6.5, 3.8, 'V',   'S17 Transf. Biomed'),
    (18, 7.0, 6.0, 'V',   'S18 VAE/GAN'),
    (19, 7.5, 6.5, 'V',   'S19 Difusión/LLM'),
    (20, 6.8, 5.5, 'V',   'S20 SSL'),
    (21, 5.5, 2.0, 'VI',  'S21 Ética'),
    (22, 5.0, 2.5, 'VI',  'S22 Cierre'),
]

colores_modulo_mapa = {
    'I': '#5B9BD5', 'II': '#ED7D31', 'III': '#70AD47',
    'IV': '#FFC000', 'V': '#FF0000', 'VI': '#7030A0'
}

fig, ax = plt.subplots(figsize=(13, 8))

# Dibujar regiones de fondo claras
ax.fill_between([0, 4.5], [0, 0], [8, 8], alpha=0.04, color='steelblue')
ax.fill_between([4.5, 8], [0, 0], [8, 8], alpha=0.04, color='tomato')
ax.text(2.2, 7.5, 'ML Clásico', fontsize=9, color='steelblue', alpha=0.7, style='italic')
ax.text(5.8, 7.5, 'Deep Learning', fontsize=9, color='tomato',    alpha=0.7, style='italic')

# Graficar sesiones
for sess, x, y, mod, label in mapa_sesiones:
    color = colores_modulo_mapa[mod]
    ax.scatter(x, y, s=160, color=color, zorder=5, edgecolors='white', lw=1.2)
    ax.annotate(label, (x, y), fontsize=7,
                 xytext=(5, 5), textcoords='offset points', color=color)

# Leyenda de módulos
handles = [mpatches.Patch(color=colores_modulo_mapa[m], label=lbl)
           for m, lbl in [
               ('I',  'I: Fundamentos Estadísticos'),
               ('II', 'II: Modelos Discriminativos'),
               ('III','III: Generativos (Clásicos)'),
               ('IV', 'IV: Deep Learning'),
               ('V',  'V: Arquitecturas Avanzadas'),
               ('VI', 'VI: Ética y Cierre'),
           ]]
ax.legend(handles=handles, fontsize=8, loc='lower right')

ax.set(xlabel='Nivel de abstracción  (estadístico → profundo)',
       ylabel='Alcance de modalidad de datos  (tabular → señal cruda → generativo)',
       xlim=(0, 8.5), ylim=(0, 8),
       title='Mapa del curso: 22 sesiones a través del panorama del ML biomédico')
plt.tight_layout()
plt.show()

## Parte 5 — Qué llevarse

Algunos principios que atraviesan todos los métodos de este curso — escritos como
recordatorios, no como reglas.

In [ ]:
principios = [
    ('Evalúa con honestidad',
     'Usa LOSO o CV estratificada por grupo para datos fisiológicos. '
     'Un modelo que se ve bien con k-fold aleatorio pero falla en LOSO no es un modelo — '
     'es un reconocedor de sujetos.'),

    ('Empareja el método con el régimen de datos',
     'Regresión logística + AUROC en 50 muestras supera a una CNN mal regularizada. '
     'La complejidad no es progreso. Añádela solo cuando los métodos más simples fallen '
     'por una razón diagnosticable.'),

    ('Reporta la incertidumbre',
     'Un solo número sin intervalo de confianza es un resultado incompleto. '
     'Un intervalo de confianza sin su tamaño de muestra es engañoso. '
     'Publica la distribución, no el pico.'),

    ('Comprende tu métrica',
     'El AUROC es independiente de la prevalencia pero oculta la calibración. '
     'La exactitud oculta el desbalance de clases. El F1 oculta la sensibilidad al umbral. '
     'El análisis de curva de decisión es la métrica más honesta clínicamente y la '
     'menos usada — corrige eso en tu propio trabajo.'),

    ('Explica antes de desplegar',
     'Un modelo que un clínico no puede interrogar es un modelo que no usará — '
     'o peor, usará sin entender. '
     'La interpretabilidad no es un extra agradable; es un prerrequisito para la confianza.'),

    ('Piensa en quién no está en tu conjunto de entrenamiento',
     'Cada dataset es una muestra de una población. '
     'Pregunta quién fue excluido, por qué, y qué significa eso para el comportamiento '
     'de tu modelo en personas que están sistemáticamente subrrepresentadas.'),

    ('La reproducibilidad es una forma de respeto',
     'El revisor que no puede ejecutar tu código, el clínico que no puede '
     'replicar tus resultados, y el paciente afectado por tu modelo '
     'todos merecen un pipeline que puedan inspeccionar y verificar.'),

    ('Mantente actualizado, mantente humilde',
     'Los métodos de este curso quedarán parcialmente obsoletos en cinco años. '
     'El pensamiento estadístico, los hábitos de evaluación crítica, y los '
     'marcos éticos no lo harán.'),
]

print('=' * 65)
print('PRINCIPIOS PARA ML BIOMÉDICO — PARA LLEVAR ADELANTE')
print('=' * 65)
for i, (titulo, cuerpo) in enumerate(principios, 1):
    print(f'\n{i}. {titulo.upper()}')
    # Envolver el texto del cuerpo
    palabras = cuerpo.split()
    line  = '   '
    for palabra in palabras:
        if len(line) + len(palabra) > 62:
            print(line)
            line = '   ' + palabra + ' '
        else:
            line += palabra + ' '
    if line.strip():
        print(line)
print('\n' + '=' * 65)

## Especificación del proyecto final (referencia)

El proyecto final debe entregarse como un repositorio de GitHub que contenga:

1. `README.md` — dataset, planteamiento del problema, resumen de resultados, instrucciones de configuración
2. `notebooks/` — un cuaderno por cada etapa principal del pipeline
3. `src/` — módulos de Python reutilizables (preprocesamiento, modelos, evaluación)
4. `results/` — métricas guardadas, figuras, pesos del modelo entrenado (si < 100 MB)
5. `report.pdf` — 6–8 páginas siguiendo el estilo NeurIPS o MICCAI

**Componentes mínimos requeridos en el reporte:**

| Sección | Contenido requerido |
|---|---|
| Introducción | Motivación clínica, trabajo previo, contribución |
| Métodos | Dataset, preprocesamiento, modelos, protocolo de evaluación |
| Resultados | AUROC con IC 95%, tabla comparativa, curvas de aprendizaje |
| Análisis | Figura de explicabilidad, gráfico de calibración, análisis de subgrupos |
| Discusión | Limitaciones, modos de falla, trasladabilidad clínica |
| Conclusión | Resumen, direcciones futuras |

**Datasets recomendados para proyectos no comprometidos aún:**

| Dataset | Tarea | Enlace |
|---|---|---|
| PhysioNet CinC 2017 | Detección de FA desde ECG | physionet.org/content/challenge-2017 |
| CHB-MIT EEG | Detección de crisis | physionet.org/content/chbmit |
| ISRUC Sleep | Estadificación del sueño | sleeptight.isr.uc.pt |
| PhysioNet CinC 2012 | Mortalidad en UCI | physionet.org/content/challenge-2012 |
| BCI Competition IV 2a | BCI de imaginería motora | bbci.de/competition/iv |
| PTB-XL ECG | Clasificación de ECG de 12 derivaciones | physionet.org/content/ptb-xl |
| DREAMER EEG/ECG | Reconocimiento de emociones | zenodo.org/record/546113 |

---

*Este cuaderno y los materiales del curso se publican bajo GPL-3.0.*
*Por favor contribuye mejoras, correcciones, y nuevos ejemplos biomédicos al repositorio.*